In [1]:
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from huggingface_hub import HfApi

import pandas as pd
import torch
import numpy as np

# Download

In [2]:
# Training dataset
ds = load_dataset("recursionpharma/rxrx3-core")

Resolving data files:   0%|          | 0/35 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/35 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/35 [00:00<?, ?it/s]

In [3]:
# Embedding and metadata

file_path_metadata = hf_hub_download("recursionpharma/rxrx3-core", filename="metadata_rxrx3_core.csv",repo_type="dataset")
file_path_embs = hf_hub_download("recursionpharma/rxrx3-core", filename="OpenPhenom_rxrx3_core_embeddings.parquet",repo_type="dataset")

open_phenom_embeddings = pd.read_parquet(file_path_embs)
rxrx3_core_metadata = pd.read_csv(file_path_metadata)

/tmp/ipykernel_1839353/2011887431.py:7: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  rxrx3_core_metadata = pd.read_csv(file_path_metadata)


# Check data

## Training data

In [ ]:
print(ds)
print(ds["train"][:10])

## Metadata

In [ ]:
rxrx3_core_metadata

In [ ]:
# Filter dataframe to get rows where SMILES is not NaN
rxrx3_core_metadata_with_smiles = rxrx3_core_metadata[rxrx3_core_metadata['SMILES'].notna()]
rxrx3_core_metadata_with_smiles

In [ ]:
conditions = {
	"experiment_name": "compound-001",
	"plate": 1,
	"address": "AA15"
}
def ds_condition(ds, conditions):
	mask = ds[list(conditions)].eq(pd.Series(conditions)).all(axis=1)
	filtered = ds[mask]
	return filtered

metadata_filtered = ds_condition(rxrx3_core_metadata, conditions)
metadata_filtered

## Embeddings

In [ ]:
open_phenom_embeddings

In [ ]:
well_condition = {
	"well_id": "compound-001_1_AA15"
}

embeddings_filtered = ds_condition(open_phenom_embeddings, well_condition)
embeddings_filtered

# Process data

## Merge SMILES and embedding

In [ ]:
meta = rxrx3_core_metadata_with_smiles
emb  = open_phenom_embeddings
meta_non_nan = meta[meta["SMILES"].notna()].copy()

In [ ]:
merged = meta_non_nan.merge(
    emb,
    on="well_id",
    how="inner",            # only wells present in both
    # validate="one_to_one" # or "many_to_one" if you expect duplicates
)

In [ ]:
merged

In [ ]:
print(f"Number of unique SMILES: {merged['SMILES'].nunique()}")
print(f"Number of unique well_ids: {merged['well_id'].nunique()}")
# print(f"Number of unique experiments: {merged['experiment_name'].nunique()}")
print(f"Number of unique plates: {merged['plate'].nunique()}")
print(f"Number of unique addresses: {merged['address'].nunique()}")


In [ ]:
merged_df = merged.copy()
merged_df.to_csv("rxrx3_smiles_embeddings.csv", index=False)

## Hugging face

In [ ]:
api = HfApi()
api.upload_file(
    path_or_fileobj="rxrx3_smiles_embeddings.csv",
    path_in_repo="rxrx3_smiles_embeddings.csv",
    repo_id="hyunnnnnnnn/rxrx3_smiles_embedding",
    repo_type="dataset",
)

In [ ]:
merged_df.to_parquet("rxrx3_smiles_embeddings.parquet", index=False)

In [ ]:
api.upload_file(
    path_or_fileobj="rxrx3_smiles_embeddings.parquet",
    path_in_repo="rxrx3_smiles_embeddings.parquet",
    repo_id="hyunnnnnnnn/rxrx3_smiles_embedding",
    repo_type="dataset",
)

In [2]:
path = hf_hub_download(
    "hyunnnnnnnn/rxrx3_smiles_embedding",
    "rxrx3_smiles_embeddings.csv",
    repo_type="dataset"
)
df_csv = pd.read_csv(path)

In [3]:
df_parquet = pd.read_parquet(
    hf_hub_download(
        "hyunnnnnnnn/rxrx3_smiles_embedding",
        "rxrx3_smiles_embeddings.parquet",
        repo_type="dataset"
    )
)

In [4]:
df_parquet

,well_id,experiment_name,plate,address,gene,treatment,SMILES,concentration,perturbation_type,cell_type,...,feature_374,feature_375,feature_376,feature_377,feature_378,feature_379,feature_380,feature_381,feature_382,feature_383
0,compound-003_11_AD37,compound-003,11,AD37,None,Phloretin,"OC1=CC=C(CCC(=O)C2=C(O)C=C(O)C=C2O)C=C1 |c:9,1...",0.0250,COMPOUND,HUVEC,...,-0.013445,-0.162979,0.054389,-0.069481,-0.124423,0.022028,0.022953,-0.079113,0.173631,-0.128588
1,compound-003_35_Y15,compound-003,35,Y15,None,Clozapine,CN1CCN(CC1)C1=NC2=C(NC3=C1C=CC=C3)C=CC(Cl)=C2 ...,2.5000,COMPOUND,HUVEC,...,0.017443,-0.176950,0.068680,-0.112014,-0.106931,-0.081489,0.027819,-0.096354,0.191950,-0.200426
2,compound-001_19_D20,compound-001,19,D20,None,Dequalinium,CC1=[N+](CCCCCCCCCC[N+]2=C(C)C=C(N)C3=CC=CC=C2...,0.2500,COMPOUND,HUVEC,...,0.047074,-0.097213,0.044904,-0.060549,-0.114342,-0.056257,0.036089,-0.065139,0.229080,-0.172424
3,compound-001_11_AD07,compound-001,11,AD07,None,ethotoin,"CCN1C(=O)NC(C1=O)C1=CC=CC=C1 |c:12,14,t:10|",0.0025,COMPOUND,HUVEC,...,0.031462,-0.124591,0.044777,-0.106306,-0.129522,-0.061542,0.035425,-0.105501,0.197642,-0.169117
4,compound-003_35_D11,compound-003,35,D11,None,nitenpyram,CCN(CC1=CN=C(Cl)C=C1)C(\NC)=C\[N+]([O-])=O |c:...,1.0000,COMPOUND,HUVEC,...,-0.020100,-0.168675,0.065233,-0.103493,-0.125883,-0.053245,0.037317,-0.093387,0.206744,-0.161665
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63400,compound-003_33_F04,compound-003,33,F04,None,cefonicid,O[C@@H](C(=O)NC1[C@H]2SCC(CSC3=NN=NN3CS(O)(=O)...,1.0000,COMPOUND,HUVEC,...,0.039692,-0.161012,0.072920,-0.075982,-0.111072,-0.008118,0.012141,-0.059071,0.157921,-0.177434
63401,compound-003_44_AA03,compound-003,44,AA03,None,timonacic,OC(=O)C1CSCN1,0.0025,COMPOUND,HUVEC,...,-0.049750,-0.144031,0.027755,-0.100220,-0.090184,-0.040595,0.096588,-0.052413,0.206018,-0.189288
63402,compound-004_38_AB45,compound-004,38,AB45,None,Chenodeoxycholic acid,[H][C@]1(CC[C@@]2([H])[C@]3([H])[C@H](O)C[C@]4...,0.0100,COMPOUND,HUVEC,...,-0.045291,-0.173744,0.064587,-0.150534,-0.093916,-0.050701,0.050078,-0.096220,0.237417,-0.108089
63403,compound-004_34_L42,compound-004,34,L42,None,Pamoic acid disodium salt,OC1=C(CC2=C3C=CC=CC3=CC(C([O-])=O)=C2O)C2=CC=C...,10.0000,COMPOUND,HUVEC,...,-0.097415,-0.129583,0.029747,-0.244003,-0.129945,0.008000,0.213724,-0.029371,0.236853,-0.038210


In [14]:
df_parquet['SMILES'][0]

'OC1=CC=C(CCC(=O)C2=C(O)C=C(O)C=C2O)C=C1 |c:9,15,19,t:1,3,12|'

## Pytorch data

In [6]:
df = df_parquet
df = df[df["SMILES"].notna()].reset_index(drop=True)

embedding_cols = [c for c in df.columns if c.startswith("feature_")]
print(len(embedding_cols))   # should be 384

384


In [9]:
embeddings_np = df[embedding_cols].to_numpy(dtype=np.float32)   # (N, 384)
embeddings = torch.from_numpy(embeddings_np)                    # torch.float32 tensor
print(embeddings.shape)

torch.Size([63405, 384])


In [15]:
df["SMILES_clean"] = df["SMILES"].str.split("|").str[0]
smiles_list = df["SMILES_clean"].tolist()
well_ids    = df["well_id"].tolist()

In [17]:
save_obj = {
    "embeddings": embeddings,   # (N, 384) float32
    "smiles": smiles_list,      # list[str]
    "well_id": well_ids,        # list[str], optional
}

torch.save(save_obj, "./data/rxrx3_smiles_embeddings.pt")

In [23]:
obj = torch.load("./data/rxrx3_smiles_embeddings.pt", map_location="cpu", weights_only=True)
embeddings = obj["embeddings"]
smiles     = obj["smiles"]
well_id    = obj["well_id"]

In [26]:
api = HfApi()
api.upload_file(
    path_or_fileobj="./data/rxrx3_smiles_embeddings.pt",
    path_in_repo="rxrx3_smiles_embeddings.pt",
    repo_id="hyunnnnnnnn/rxrx3_smiles_embedding",
    repo_type="dataset",
)

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/hyunnnnnnnn/rxrx3_smiles_embedding/commit/e821be4bcd6ab6ebe254ccde36fde0b2ece58991', commit_message='Upload rxrx3_smiles_embeddings.pt with huggingface_hub', commit_description='', oid='e821be4bcd6ab6ebe254ccde36fde0b2ece58991', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/hyunnnnnnnn/rxrx3_smiles_embedding', endpoint='https://huggingface.co', repo_type='dataset', repo_id='hyunnnnnnnn/rxrx3_smiles_embedding'), pr_revision=None, pr_num=None)

## Check data

In [4]:
path = hf_hub_download(
    "hyunnnnnnnn/rxrx3_smiles_embedding",
    "rxrx3_smiles_embeddings.csv",
    repo_type="dataset"
)
df_csv = pd.read_csv(path)

In [5]:
df_csv

,well_id,experiment_name,plate,address,gene,treatment,SMILES,concentration,perturbation_type,cell_type,...,feature_374,feature_375,feature_376,feature_377,feature_378,feature_379,feature_380,feature_381,feature_382,feature_383
0,compound-003_11_AD37,compound-003,11,AD37,NaN,Phloretin,"OC1=CC=C(CCC(=O)C2=C(O)C=C(O)C=C2O)C=C1 |c:9,1...",0.0250,COMPOUND,HUVEC,...,-0.013445,-0.162979,0.054389,-0.069481,-0.124423,0.022028,0.022953,-0.079113,0.173631,-0.128588
1,compound-003_35_Y15,compound-003,35,Y15,NaN,Clozapine,CN1CCN(CC1)C1=NC2=C(NC3=C1C=CC=C3)C=CC(Cl)=C2 ...,2.5000,COMPOUND,HUVEC,...,0.017443,-0.176950,0.068680,-0.112014,-0.106931,-0.081489,0.027819,-0.096354,0.191950,-0.200426
2,compound-001_19_D20,compound-001,19,D20,NaN,Dequalinium,CC1=[N+](CCCCCCCCCC[N+]2=C(C)C=C(N)C3=CC=CC=C2...,0.2500,COMPOUND,HUVEC,...,0.047074,-0.097213,0.044904,-0.060549,-0.114342,-0.056257,0.036089,-0.065139,0.229080,-0.172424
3,compound-001_11_AD07,compound-001,11,AD07,NaN,ethotoin,"CCN1C(=O)NC(C1=O)C1=CC=CC=C1 |c:12,14,t:10|",0.0025,COMPOUND,HUVEC,...,0.031462,-0.124591,0.044777,-0.106306,-0.129522,-0.061542,0.035425,-0.105501,0.197642,-0.169117
4,compound-003_35_D11,compound-003,35,D11,NaN,nitenpyram,CCN(CC1=CN=C(Cl)C=C1)C(\NC)=C\[N+]([O-])=O |c:...,1.0000,COMPOUND,HUVEC,...,-0.020100,-0.168675,0.065233,-0.103493,-0.125883,-0.053245,0.037317,-0.093387,0.206744,-0.161665
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63400,compound-003_33_F04,compound-003,33,F04,NaN,cefonicid,O[C@@H](C(=O)NC1[C@H]2SCC(CSC3=NN=NN3CS(O)(=O)...,1.0000,COMPOUND,HUVEC,...,0.039692,-0.161012,0.072920,-0.075982,-0.111072,-0.008118,0.012141,-0.059071,0.157921,-0.177434
63401,compound-003_44_AA03,compound-003,44,AA03,NaN,timonacic,OC(=O)C1CSCN1,0.0025,COMPOUND,HUVEC,...,-0.049750,-0.144031,0.027755,-0.100220,-0.090184,-0.040595,0.096588,-0.052413,0.206018,-0.189288
63402,compound-004_38_AB45,compound-004,38,AB45,NaN,Chenodeoxycholic acid,[H][C@]1(CC[C@@]2([H])[C@]3([H])[C@H](O)C[C@]4...,0.0100,COMPOUND,HUVEC,...,-0.045291,-0.173744,0.064587,-0.150534,-0.093916,-0.050701,0.050078,-0.096220,0.237417,-0.108089
63403,compound-004_34_L42,compound-004,34,L42,NaN,Pamoic acid disodium salt,OC1=C(CC2=C3C=CC=CC3=CC(C([O-])=O)=C2O)C2=CC=C...,10.0000,COMPOUND,HUVEC,...,-0.097415,-0.129583,0.029747,-0.244003,-0.129945,0.008000,0.213724,-0.029371,0.236853,-0.038210


In [8]:
pd.read_csv("./data/rxrx3_smiles_embeddings.csv")

,well_id,experiment_name,plate,address,gene,treatment,SMILES,concentration,perturbation_type,cell_type,...,feature_374,feature_375,feature_376,feature_377,feature_378,feature_379,feature_380,feature_381,feature_382,feature_383
0,compound-003_11_AD37,compound-003,11,AD37,NaN,Phloretin,"OC1=CC=C(CCC(=O)C2=C(O)C=C(O)C=C2O)C=C1 |c:9,1...",0.0250,COMPOUND,HUVEC,...,-0.013445,-0.162979,0.054389,-0.069481,-0.124423,0.022028,0.022953,-0.079113,0.173631,-0.128588
1,compound-003_35_Y15,compound-003,35,Y15,NaN,Clozapine,CN1CCN(CC1)C1=NC2=C(NC3=C1C=CC=C3)C=CC(Cl)=C2 ...,2.5000,COMPOUND,HUVEC,...,0.017443,-0.176950,0.068680,-0.112014,-0.106931,-0.081489,0.027819,-0.096354,0.191950,-0.200426
2,compound-001_19_D20,compound-001,19,D20,NaN,Dequalinium,CC1=[N+](CCCCCCCCCC[N+]2=C(C)C=C(N)C3=CC=CC=C2...,0.2500,COMPOUND,HUVEC,...,0.047074,-0.097213,0.044904,-0.060549,-0.114342,-0.056257,0.036089,-0.065139,0.229080,-0.172424
3,compound-001_11_AD07,compound-001,11,AD07,NaN,ethotoin,"CCN1C(=O)NC(C1=O)C1=CC=CC=C1 |c:12,14,t:10|",0.0025,COMPOUND,HUVEC,...,0.031462,-0.124591,0.044777,-0.106306,-0.129522,-0.061542,0.035425,-0.105501,0.197642,-0.169117
4,compound-003_35_D11,compound-003,35,D11,NaN,nitenpyram,CCN(CC1=CN=C(Cl)C=C1)C(\NC)=C\[N+]([O-])=O |c:...,1.0000,COMPOUND,HUVEC,...,-0.020100,-0.168675,0.065233,-0.103493,-0.125883,-0.053245,0.037317,-0.093387,0.206744,-0.161665
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63400,compound-003_33_F04,compound-003,33,F04,NaN,cefonicid,O[C@@H](C(=O)NC1[C@H]2SCC(CSC3=NN=NN3CS(O)(=O)...,1.0000,COMPOUND,HUVEC,...,0.039692,-0.161012,0.072920,-0.075982,-0.111072,-0.008118,0.012141,-0.059071,0.157921,-0.177434
63401,compound-003_44_AA03,compound-003,44,AA03,NaN,timonacic,OC(=O)C1CSCN1,0.0025,COMPOUND,HUVEC,...,-0.049750,-0.144031,0.027755,-0.100220,-0.090184,-0.040595,0.096588,-0.052413,0.206018,-0.189288
63402,compound-004_38_AB45,compound-004,38,AB45,NaN,Chenodeoxycholic acid,[H][C@]1(CC[C@@]2([H])[C@]3([H])[C@H](O)C[C@]4...,0.0100,COMPOUND,HUVEC,...,-0.045291,-0.173744,0.064587,-0.150534,-0.093916,-0.050701,0.050078,-0.096220,0.237417,-0.108089
63403,compound-004_34_L42,compound-004,34,L42,NaN,Pamoic acid disodium salt,OC1=C(CC2=C3C=CC=CC3=CC(C([O-])=O)=C2O)C2=CC=C...,10.0000,COMPOUND,HUVEC,...,-0.097415,-0.129583,0.029747,-0.244003,-0.129945,0.008000,0.213724,-0.029371,0.236853,-0.038210


In [9]:
# Generate the list of column names
cols = [f"feature_{i}" for i in range(384)]

# Print it in YAML-compatible format
print(str(cols).replace("'", '"'))

["feature_0", "feature_1", "feature_2", "feature_3", "feature_4", "feature_5", "feature_6", "feature_7", "feature_8", "feature_9", "feature_10", "feature_11", "feature_12", "feature_13", "feature_14", "feature_15", "feature_16", "feature_17", "feature_18", "feature_19", "feature_20", "feature_21", "feature_22", "feature_23", "feature_24", "feature_25", "feature_26", "feature_27", "feature_28", "feature_29", "feature_30", "feature_31", "feature_32", "feature_33", "feature_34", "feature_35", "feature_36", "feature_37", "feature_38", "feature_39", "feature_40", "feature_41", "feature_42", "feature_43", "feature_44", "feature_45", "feature_46", "feature_47", "feature_48", "feature_49", "feature_50", "feature_51", "feature_52", "feature_53", "feature_54", "feature_55", "feature_56", "feature_57", "feature_58", "feature_59", "feature_60", "feature_61", "feature_62", "feature_63", "feature_64", "feature_65", "feature_66", "feature_67", "feature_68", "feature_69", "feature_70", "feature_71", "